# kNN 如何做商品分类？

**面试回答：**kNN 保存样本，预测时按距离取 k 个邻居投票；它的效果由距离、缩放与 k 决定，训练快不代表查询快。

## 真实案例

新商品要根据价格和浏览秒数补齐品类，距离必须先消除量纲差异。

In [1]:
import numpy as np  # 导入 NumPy 手写 kNN。
name=np.array(['咖啡豆','滤纸','瑜伽垫','水壶','新咖啡豆','新水壶'])  # 构造商品名称。
x=np.array([[80,230],[18,180],[90,35],[40,30],[92,220],[45,38]],dtype=float)  # 记录价格和浏览时长。
y=np.array([0,0,1,1,0,1])  # 标记咖啡零类与运动一类。
print('商品 | 价格 | 浏览秒 | 类别')  # 输出商品表头。
for n,row,c in zip(name,x,y):  # 逐条展示商品样本。
    print(n,row.tolist(),c)  # 输出一条商品记录。

商品 | 价格 | 浏览秒 | 类别
咖啡豆 [80.0, 230.0] 0
滤纸 [18.0, 180.0] 0
瑜伽垫 [90.0, 35.0] 1
水壶 [40.0, 30.0] 1
新咖啡豆 [92.0, 220.0] 0
新水壶 [45.0, 38.0] 1


## Baseline / 基线

基线只按价格阈值分类，忽略浏览时长。

In [2]:
train=np.arange(4)  # 定义历史商品索引。
valid=np.arange(4,6)  # 定义新商品索引。
baseline=(x[valid,0]<60).astype(int)  # 用价格阈值生成错误的硬规则。
baseline_acc=float(np.mean(baseline==y[valid]))  # 计算基线准确率。
print('价格基线:',baseline.tolist(),'准确率=',baseline_acc)  # 输出基线结果。

价格基线: [0, 1] 准确率= 1.0


In [3]:
mean=x[train].mean(axis=0)  # 只用历史商品计算均值。
std=x[train].std(axis=0)  # 只用历史商品计算标准差。
z_train=(x[train]-mean)/std  # 标准化历史商品。
z_valid=(x[valid]-mean)/std  # 用训练统计量标准化新商品。
def knn(query,k):  # 定义手写最近邻投票函数。
    distance=((z_train-query)**2).sum(axis=1)  # 计算到历史商品的平方距离。
    near=distance.argsort()[:k]  # 选出最近的 k 个索引。
    vote=y[train][near].mean()  # 用邻居类别均值进行投票。
    return int(vote>=.5),near,distance  # 返回类别、邻居和距离。
result=[knn(q,3) for q in z_valid]  # 对两件新商品执行 k=3 分类。
pred=np.array([r[0] for r in result])  # 提取预测类别。
acc=float(np.mean(pred==y[valid]))  # 计算 kNN 准确率。
print('首个查询距离:',np.round(result[0][2],2),'邻居:',name[train][result[0][1]].tolist())  # 输出距离中间量。

首个查询距离: [0.18 6.6  4.42 7.81] 邻居: ['咖啡豆', '瑜伽垫', '滤纸']


## 结果解读

k=1 易记住噪声，k 增大更平滑却可能跨越局部边界；距离加权可进一步降低远邻影响。

In [4]:
print('商品 | 真实 | kNN预测 | 邻居')  # 输出结果表头。
for local,index in enumerate(valid):  # 逐条展示新商品预测。
    print(name[index],y[index],pred[local],name[train][result[local][1]].tolist())  # 输出邻居证据。
print('生产差距：需向量索引、过滤条件、缓存、近重复去重和召回延迟监控。')  # 描述生产差距。

商品 | 真实 | kNN预测 | 邻居
新咖啡豆 0 0 ['咖啡豆', '瑜伽垫', '滤纸']
新水壶 1 1 ['水壶', '瑜伽垫', '滤纸']
生产差距：需向量索引、过滤条件、缓存、近重复去重和召回延迟监控。


## 失败案例与修复

未缩放时价格主导距离；修复是仅用训练集拟合缩放参数。

In [5]:
raw_distance=((x[train]-x[valid][0])**2).sum(axis=1)  # 计算未缩放距离。
raw_neighbor=raw_distance.argmin()  # 选出未缩放最近商品。
print('失败：未缩放首邻居=',name[train][raw_neighbor])  # 输出量纲主导的结果。
print('修复：标准化邻居=',name[train][result[0][1]].tolist())  # 输出缩放后的邻居。
print('提示：高维空间距离会集中，精确 kNN 需要重新评估。')  # 说明维度灾难。

失败：未缩放首邻居= 咖啡豆
修复：标准化邻居= ['咖啡豆', '瑜伽垫', '滤纸']
提示：高维空间距离会集中，精确 kNN 需要重新评估。


In [6]:
assert len(name)>=5  # 保护商品样本数。
assert acc>=baseline_acc  # 保护 kNN 不弱于规则基线。
assert len(result[0][1])==3  # 保护实际使用三个邻居。
assert not np.allclose(raw_distance,result[0][2])  # 保护缩放改变距离几何。